# Dataset Creation
This notebook:
- load correlation matrices from a `.pt` file
- select only matrices in the range `[MATRIX_START, MATRIX_END)`
- split matrices into train, validation, and test sets
- save the three datasets under `data/processed/dataset`

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import torch
import math

np.set_printoptions(suppress=True, precision=4)

## Step 1: Load Data and Select Matrix Range
Load correlation matrices from a `.pt` file and keep only matrices in `[MATRIX_START, MATRIX_END)` (same behavior as `05_linearAE.ipynb`).

In [2]:
FILE_NAME = 'data_00_20'
WINDOW_SIZE = 252
STRIDE = 5
MY_FILE_NAME = f'{FILE_NAME}_w{WINDOW_SIZE}_s{STRIDE}.pt'

if 'google.colab' in sys.modules:
    print('Environment detected: Google Colab')
    IS_COLAB = True
else:
    print('Environment detected: Local (PC)')
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    processed_root = Path('/content/drive/MyDrive/dataset_tesi')
else:
    project_root = Path.cwd().resolve().parent
    processed_root = project_root / 'data' / 'processed' / FILE_NAME

corr_matrices_dir = processed_root / 'correlation_matrices'
if not corr_matrices_dir.exists():
    corr_matrices_dir = processed_root

selected_file = corr_matrices_dir / MY_FILE_NAME
if not selected_file.exists():
    raise FileNotFoundError(f"File '{MY_FILE_NAME}' not found in: {corr_matrices_dir.resolve()}")

# Select the matrix range to use: [MATRIX_START, MATRIX_END)
MATRIX_START = 37  # number or None
MATRIX_END = None  # number or None; e.g., 500 to use only first 500 matrices

print(f'Selected file: {selected_file.name}')
print(f'Matrix range requested: [{MATRIX_START}, {MATRIX_END})')

Environment detected: Local (PC)
Selected file: data_00_20_w252_s5.pt
Matrix range requested: [37, None)


In [3]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')

    if isinstance(payload, dict):
        corr_tensor = payload.get('corr_tensor', None)
        meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}
    elif isinstance(payload, torch.Tensor):
        corr_tensor = payload
        meta = {}
    else:
        raise TypeError(f'Unsupported .pt format: {type(payload)}')

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')

    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')

    return corr_tensor.float(), meta

# --- CARICAMENTO ---
corr_tensor, meta = load_corr_payload(selected_file)

orig_n = corr_tensor.shape[0]
start_idx = 0 if MATRIX_START is None else int(MATRIX_START)
end_idx = orig_n if MATRIX_END is None else int(MATRIX_END)

if not (0 <= start_idx < end_idx <= orig_n):
    raise ValueError(f'Invalid matrix range [{start_idx}, {end_idx}) for dataset size {orig_n}')

# --- TAGLIO DEI DATI ---
corr_tensor = corr_tensor[start_idx:end_idx]

if 'window_ranges' in meta and len(meta['window_ranges']) == orig_n:
    meta['window_ranges'] = meta['window_ranges'][start_idx:end_idx]

# ==========================================
# ESTRAZIONE METADATI (Timestamps, ecc.)
# ==========================================
# Estraiamo in variabili "sciolte" così lo script di partizionamento le trova
timestamps = meta.get('timestamps', [])
window_ranges = meta.get('window_ranges', [])
window_length = meta.get('window_length', 252)
stride = meta.get('stride', 5)


# --- OUTPUT DI CONTROLLO ---
print(f"{'='*50}")
print("DATASET LOADED & SLICED")
print(f"{'='*50}")
print(f'corr_tensor shape : {tuple(corr_tensor.shape)}')
print(f'using matrices    : [{start_idx}, {end_idx}) out of {orig_n}')
print(f'dtype             : {corr_tensor.dtype}')

# Se abbiamo i timestamp, stampiamo la prima e l'ultima data per controllo
if timestamps and window_ranges:
    first_matrix_start = timestamps[window_ranges[0][0]]
    first_matrix_end   = timestamps[window_ranges[0][1] - 1]
    last_matrix_start  = timestamps[window_ranges[-1][0]]
    last_matrix_end    = timestamps[window_ranges[-1][1] - 1]
    
    print("-" * 50)
    print(f"Total timestamps  : {len(timestamps)}")
    print(f"First matrix date : {first_matrix_start} to {first_matrix_end}")
    print(f"Last matrix date  : {last_matrix_start} to {last_matrix_end}")
else:
    print("-" * 50)
    print("Warning: 'timestamps' or 'window_ranges' missing from .pt file metadata!")
print(f"{'='*50}\n")

DATASET LOADED & SLICED
corr_tensor shape : (980, 100, 100)
using matrices    : [37, 1017) out of 1017
dtype             : torch.float32
--------------------------------------------------
Total timestamps  : 5333
First matrix date : 2000-07-27 to 2001-07-26
Last matrix date  : 2020-01-13 to 2021-01-11



## Step 2: Create Train/Validation/Test Splits

In [4]:
# --- PARAMETRI ---
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
WINDOW_SIZE = 252  # Dimensione temporale della finestra (usata per il controllo logico)
STRIDE = 5         # Passo tra una matrice e l'altra in giorni

# --- NUOVO PARAMETRO FINANZIARIO ---
FORWARD_DAYS = 21  # Orizzonte di Asset Allocation (gap tra train-val e val-test)

# Verifica delle frazioni
if abs(TRAIN_FRACTION + VAL_FRACTION + TEST_FRACTION - 1.0) > 1e-6:
    raise ValueError("Le frazioni di Train, Val e Test devono sommare a 1.0")

# Setup iniziale
corr_np = corr_tensor.numpy().astype(np.float32)
n_matrices, n_assets, _ = corr_np.shape

# ==========================================
# PARTIZIONAMENTO CON GAP DI 1 MESE
# ==========================================

# Calcoliamo quante matrici compongono il gap di 21 giorni (ceil per sicurezza)
GAP_MATRICES = math.ceil(FORWARD_DAYS / STRIDE) # 21 / 5 = 4.2 -> 5 matrici

# 1. Calcoliamo le matrici "utilizzabili" rimuovendo solo i due piccoli gap da 1 mese
usable_matrices = n_matrices - (2 * GAP_MATRICES)

if usable_matrices <= 0:
    raise ValueError(f"Dataset troppo piccolo ({n_matrices}) per supportare due gap di {GAP_MATRICES} matrici.")

# 2. Assegniamo le dimensioni nette
n_train = int(usable_matrices * TRAIN_FRACTION)
n_val = int(usable_matrices * VAL_FRACTION)
n_test = usable_matrices - n_train - n_val 

# 3. Costruiamo gli indici sequenziali
# --- TRAIN SET ---
train_end = n_train
train_idx = np.arange(0, train_end)

# --- VAL SET ---
val_start = train_end + GAP_MATRICES
val_end = val_start + n_val
val_idx = np.arange(val_start, val_end)

# --- TEST SET ---
test_start = val_end + GAP_MATRICES
test_idx = np.arange(test_start, n_matrices)

# 4. Estrazione dei tensori finali
train_corr = torch.from_numpy(corr_np[train_idx])
val_corr = torch.from_numpy(corr_np[val_idx])
test_corr = torch.from_numpy(corr_np[test_idx])

# ==========================================
# ESTRAZIONE DATE PER I PRINT
# ==========================================
train_start_date = timestamps[window_ranges[train_idx[0]][0]]
train_end_date = timestamps[window_ranges[train_idx[-1]][1] - 1]

val_start_date = timestamps[window_ranges[val_idx[0]][0]]
val_end_date = timestamps[window_ranges[val_idx[-1]][1] - 1]

test_start_date = timestamps[window_ranges[test_idx[0]][0]]
test_end_date = timestamps[window_ranges[test_idx[-1]][1] - 1]

# --- OUTPUT STATISTICHE ---
print(f"{'='*90}")
print("DATASET SPLIT")
print(f"{'='*90}")
print(f"Total historical matrices    : {n_matrices}")
print(f"Forward horizon (Gap)        : {FORWARD_DAYS} days ({GAP_MATRICES} matrices)")
print(f"Total dropped in gaps        : {2 * GAP_MATRICES} matrices")
print(f"Usable matrices for ML       : {usable_matrices}")
print("-" * 90)
print(f"Train shape: {str(tuple(train_corr.shape)):<14} | Idx: {train_idx[0]:4d} to {train_idx[-1]:4d} | Dates: {train_start_date} to {train_end_date}")
print(f"Val shape  : {str(tuple(val_corr.shape)):<14} | Idx: {val_idx[0]:4d} to {val_idx[-1]:4d} | Dates: {val_start_date} to {val_end_date}")
print(f"Test shape : {str(tuple(test_corr.shape)):<14} | Idx: {test_idx[0]:4d} to {test_idx[-1]:4d} | Dates: {test_start_date} to {test_end_date}")
print(f"{'='*90}")

DATASET SPLIT
Total historical matrices    : 980
Forward horizon (Gap)        : 21 days (5 matrices)
Total dropped in gaps        : 10 matrices
Usable matrices for ML       : 970
------------------------------------------------------------------------------------------
Train shape: (679, 100, 100) | Idx:    0 to  678 | Dates: 2000-07-27 to 2015-01-20
Val shape  : (145, 100, 100) | Idx:  684 to  828 | Dates: 2014-03-05 to 2018-01-10
Test shape : (146, 100, 100) | Idx:  834 to  979 | Dates: 2017-02-24 to 2021-01-11


## Step 3: Save Train/Validation/Test Datasets
Save the `all.pt` file with all matrices in original index order, then save the three split `.pt` files and the summary `.json` inside a dedicated subfolder under `data/processed/{FILE_NAME}/dataset`.

In [5]:
dataset_root_dir = processed_root / 'dataset'
dataset_root_dir.mkdir(parents=True, exist_ok=True)

range_tag = f'range_{start_idx}_{end_idx}'
split_tag = f'train{int(TRAIN_FRACTION * 100)}_val{int(VAL_FRACTION * 100)}_test{int(TEST_FRACTION * 100)}'
base_name = selected_file.stem
output_base_name = str(FORWARD_DAYS) + "_days_gap"
dataset_dir = dataset_root_dir / base_name / output_base_name
dataset_dir.mkdir(parents=True, exist_ok=True)

CHOLESKY_JITTER = 1e-6

def compute_cholesky_lower(batch: torch.Tensor, jitter: float):
    if jitter > 0.0:
        eye = torch.eye(batch.shape[-1], dtype=batch.dtype, device=batch.device)
        batch = batch + jitter * eye
    L, info = torch.linalg.cholesky_ex(batch)
    if torch.any(info != 0):
        bad = int((info != 0).sum().item())
        raise RuntimeError(
            f"Cholesky failed for {bad} matrices. Try increasing CHOLESKY_JITTER."
        )
    return L

cholesky_all = compute_cholesky_lower(corr_tensor, CHOLESKY_JITTER)
cholesky_train = compute_cholesky_lower(train_corr, CHOLESKY_JITTER)
cholesky_val = compute_cholesky_lower(val_corr, CHOLESKY_JITTER)
cholesky_test = compute_cholesky_lower(test_corr, CHOLESKY_JITTER)

all_indices = np.arange(start_idx, end_idx)
all_payload = {
    'corr_tensor': corr_tensor.clone(),
    'cholesky_L': cholesky_all.clone(),
    'indices': all_indices.tolist(),
    'split': 'all',
}

split_payloads = {
    'train': {
        'corr_tensor': train_corr.clone(),
        'cholesky_L': cholesky_train.clone(),
        'indices': train_idx.tolist(),
    },
    'val': {
        'corr_tensor': val_corr.clone(),
        'cholesky_L': cholesky_val.clone(),
        'indices': val_idx.tolist(),
    },
    'test': {
        'corr_tensor': test_corr.clone(),
        'cholesky_L': cholesky_test.clone(),
        'indices': test_idx.tolist(),
    },
}

common_meta = {
    'source_file': str(selected_file),
    'matrix_range': {'start_idx': int(start_idx), 'end_idx': int(end_idx)},
    'gap_between_sets': str(FORWARD_DAYS)+' days',
    'matrix_shape': [int(n_assets), int(n_assets)],
    'split_fractions': {'train_fraction': float(TRAIN_FRACTION), 'val_fraction': float(VAL_FRACTION), 'test_fraction': float(TEST_FRACTION)},
    'base_name': base_name,
    'range_tag': range_tag,
    'split_tag': split_tag,
    'cholesky_jitter': float(CHOLESKY_JITTER),
    'tickers': meta.get('tickers', None),
}

saved_paths = {}
all_path = dataset_dir / 'all.pt'
torch.save(
    {
        **all_payload,
        'meta': {
            **common_meta,
            'split_size': int(all_payload['corr_tensor'].shape[0]),
        },
    },
    all_path,
)
saved_paths['all'] = all_path

for split_name, payload in split_payloads.items():
    output_path = dataset_dir / f'{split_name}.pt'
    torch.save(
        {
            **payload,
            'split': split_name,
            'meta': {
                **common_meta,
                'split_size': int(payload['corr_tensor'].shape[0]),
            },
        },
        output_path,
    )
    saved_paths[split_name] = output_path

summary_path = dataset_dir / 'dataset_info.json'
summary_payload = {
    'base_name': base_name,
    'dataset_dir': str(dataset_dir),
    'files': {k: str(v) for k, v in saved_paths.items()},
    'sizes': {
        'all': int(corr_tensor.shape[0]),
        'train': int(len(train_idx)),
        'val': int(len(val_idx)),
        'test': int(len(test_idx)),
    },
    'meta': common_meta,
}

with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_payload, f, indent=4)

print('Saved dataset files inside folder:')
for split_name, output_path in saved_paths.items():
    print(f' - {split_name}: {output_path}')
print(f' - summary: {summary_path}')

Saved dataset files inside folder:
 - all: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\data_00_20\dataset\data_00_20_w252_s5\21_days_gap\all.pt
 - train: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\data_00_20\dataset\data_00_20_w252_s5\21_days_gap\train.pt
 - val: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\data_00_20\dataset\data_00_20_w252_s5\21_days_gap\val.pt
 - test: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\data_00_20\dataset\data_00_20_w252_s5\21_days_gap\test.pt
 - summary: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\data_00_20\dataset\data_00_20_w252_s5\21_days_gap\dataset_info.json
